# EfficientNet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，EfficientNetを用いてCIFAR100データセットの100クラス物体認識を行う．ネットワークの深さ・幅・入力解像度をバランス良く同時にスケールさせる「Compound Scaling」の考え方と，MBConv（Squeeze-and-Excitationを組み込んだInverted Residual Block）の構造について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．クラス数100のカラー画像データセットです．学習用データには`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## EfficientNetとCompound Scaling
EfficientNetは，2019年に提案されたモデルであり，ネットワークの「深さ（層数）」「幅（チャンネル数）」「入力解像度」という3つの要素を，個別にではなくバランスを保ちながら同時にスケールアップする**Compound Scaling**という考え方を導入したことで知られています．

それまでの多くのモデルは，深さ・幅・解像度のいずれか1つだけを大きくしてモデルを拡大する場合がほとんどでしたが，EfficientNetの著者らは，3つを一定の比率で同時にスケールさせることで，同じ計算量（FLOPs）の増加に対してより高い認識精度が得られることを示しました．具体的には，1つのスケーリング係数$\phi$に対して，

$$\text{depth}: d = \alpha^{\phi}, \quad \text{width}: w = \beta^{\phi}, \quad \text{resolution}: r = \gamma^{\phi}$$

という形で3つを同時にスケールさせます．ここで$\alpha, \beta, \gamma$は，$\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$という制約のもとで，小規模なグリッドサーチによって求められた定数です．

このスケーリングの基準となるベースネットワーク（EfficientNet-B0）自体は，Neural Architecture Search（NAS）によって自動的に探索されたアーキテクチャです．$\phi$の値を変えて$\alpha, \beta, \gamma$をB0に適用することで，B1〜B7というより大きく高精度なモデル群が得られます．本ノートブックでは，スケーリングの基準となるB0を実装します（スケーリング自体の探索やB1以降の構築は行いません）．

なお，元論文ではB0より大きなモデルの学習時に，正則化のため「Stochastic Depth」（学習時にResidual Blockをランダムにスキップする手法）も用いられていますが，本ノートブックでは実装を簡潔にするため省略します．

## MBConv（Mobile Inverted Bottleneck Convolution）
EfficientNetの各ブロックには，MobileNetV2で提案されたInverted Residual Blockをベースに，Squeeze-and-Excitation（SE）によるchannel-wiseの重み付けを組み込んだ**MBConv**が使用されています．MBConvは次の4つの処理から構成されます．

1. **Expansion**：1×1畳み込みでチャンネル数を拡張率$t$倍に増やす（$t=1$の場合は省略）．
2. **Depthwise Convolution**：チャンネルごとに独立した$k \times k$（$k=3$または$5$）の畳み込みを行い，空間方向の特徴を抽出する．
3. **Squeeze-and-Excitation (SE)**：Global Average Poolingでチャンネルごとの応答を集約し，小さな全結合層（削減率0.25）→ReLU→全結合層→Sigmoidを通して各チャンネルの重要度を計算し，特徴マップにチャンネルごとの重みを掛ける．
4. **Projection**：1×1畳み込みで元のチャンネル数に戻す（活性化関数は適用しない，linear bottleneck）．

ストライドが1かつ入力と出力のチャンネル数が等しい場合のみ，ResNetと同様に入力を出力に加算するショートカット結合を行います．また，活性化関数にはReLUではなく，$x \cdot \text{sigmoid}(x)$で定義される**Swish（SiLU）**が使用されます．

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, se_channels):
        super().__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, se_channels, kernel_size=1)
        self.act1 = nn.SiLU(inplace=True)
        self.fc2 = nn.Conv2d(se_channels, channels, kernel_size=1)
        self.act2 = nn.Sigmoid()

    def forward(self, x):
        s = self.avgpool(x)
        s = self.act1(self.fc1(s))
        s = self.act2(self.fc2(s))
        return x * s


class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio, se_ratio=0.25):
        super().__init__()
        # ストライド1かつ入出力チャンネル数が等しい場合のみ残差接続を用いる
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio
        se_channels = max(1, int(in_channels * se_ratio))

        layers = []
        if expand_ratio != 1:
            layers += [
                nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True),
            ]
        layers += [
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size, stride=stride,
                       padding=kernel_size // 2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True),
        ]
        self.expand_dwconv = nn.Sequential(*layers)

        self.se = SqueezeExcitation(hidden_dim, se_channels)

        # Projectionでは活性化関数を適用しない（linear bottleneck）
        self.project = nn.Sequential(
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        out = self.expand_dwconv(x)
        out = self.se(out)
        out = self.project(out)
        if self.use_residual:
            out = out + x
        return out

## ネットワークの構成
EfficientNet-B0は，Stemと呼ばれる最初の畳み込み層，7つのステージに分けられたMBConvの繰り返し，そして出力直前の1×1畳み込み（Head）から構成されます．各ステージは，カーネルサイズ・拡張率・出力チャンネル数・繰り返し数・ストライドが異なります．

本ノートブックでは，32×32という小さな入力画像に対応するため，オリジナル（ImageNet版，入力224×224）で5回（Stem 1回＋MBConvステージ内4回）行われるストライド2のダウンサンプリングを2回に減らしています．実装したステージ構成は次の通りです．

| ステージ | 拡張率$t$ | カーネル | 出力ch | 繰り返し数 | ストライド |
|---|---|---|---|---|---|
| Stem | - | 3×3 | 32 | 1 | 1 |
| 1 | 1 | 3×3 | 16 | 1 | 1 |
| 2 | 6 | 3×3 | 24 | 2 | 2 |
| 3 | 6 | 5×5 | 40 | 2 | 1 |
| 4 | 6 | 3×3 | 80 | 3 | 2 |
| 5 | 6 | 5×5 | 112 | 3 | 1 |
| 6 | 6 | 5×5 | 192 | 4 | 1 |
| 7 | 6 | 3×3 | 320 | 1 | 1 |
| Head | - | 1×1 | 1280 | 1 | 1 |

Headの1×1畳み込みでチャンネル数を1280に拡張したのち，Global Average Poolingで空間方向を1×1に圧縮し，全結合層でクラス数分の出力に変換します．

In [ ]:
class EfficientNetB0(nn.Module):
    # (拡張率t, 出力ch, 繰り返し数, ストライド, カーネルサイズ)
    # 32x32入力に合わせてストライド2の回数を2箇所に削減している
    stage_cfg = [
        (1, 16, 1, 1, 3),
        (6, 24, 2, 2, 3),
        (6, 40, 2, 1, 5),
        (6, 80, 3, 2, 3),
        (6, 112, 3, 1, 5),
        (6, 192, 4, 1, 5),
        (6, 320, 1, 1, 3),
    ]

    def __init__(self, n_class=100):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True),
        )

        blocks = []
        in_channels = 32
        for expand_ratio, out_channels, repeats, stride, kernel_size in self.stage_cfg:
            for i in range(repeats):
                blocks.append(MBConv(in_channels, out_channels, kernel_size,
                                      stride if i == 0 else 1, expand_ratio))
                in_channels = out_channels
        self.blocks = nn.Sequential(*blocks)

        self.head = nn.Sequential(
            nn.Conv2d(in_channels, 1280, kernel_size=1, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1280, n_class)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したネットワークを作成し，`.to(device)`によりモデルのパラメータを`device`上に配置します．最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = EfficientNetB0(n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版EfficientNet（B0）とCIFAR版（本ノートブック）の違い

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 入力解像度 | 224×224 | 32×32 |
| Stemのストライド | 2 | 1 |
| ストライド2のステージ数 | 4（ステージ2, 3, 4, 6） | 2（ステージ2, 4） |
| 合計ダウンサンプリング率 | 32倍（224→7） | 4倍（32→8） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

ステージ構成（拡張率・カーネルサイズ・チャンネル数・繰り返し数）自体はB0のものをそのまま用いていますが，上記の通りストライド2の位置・回数のみをCIFAR100の入力サイズに合わせて変更しています．

## 課題

1. 各ステージの出力チャンネル数（幅）や繰り返し数（深さ）を変更し，Compound Scalingのように幅・深さを一定の比率で同時に増やした場合と，一方だけを増やした場合とで，パラメータ数や認識精度がどのように変化するか比較しましょう．

2. SE（Squeeze-and-Excitation）を除いたMBConvを実装し，SEの有無で認識精度やパラメータ数がどの程度変化するか確認しましょう．

3. 活性化関数をSwish（SiLU）からReLUに変更して学習を行い，認識精度や学習の収束の違いを確認しましょう．